In [1]:
#import of needed packages, adding relevant cities and their coordinates into a dictionary, declaration of weather dictionary

import requests
import pandas as pd
import copy as cp
from datetime import datetime

cities_coordinates = {
    "Atlanta": {"latitude": 33.7490, "longitude": -84.3880},
    "Charlotte": {"latitude": 35.2271, "longitude": -80.8431},
    "Chicago": {"latitude": 41.8781, "longitude": -87.6298},
    "Cleveland": {"latitude": 41.4993, "longitude": -81.6944},
    "Copenhagen": {"latitude": 55.6761, "longitude": 12.5683},
    "Denver": {"latitude": 39.7392, "longitude": -104.9903},
    "Las Vegas": {"latitude": 36.1699, "longitude": -115.1398},
    "Minneapolis": {"latitude": 44.9778, "longitude": -93.2650},
    "New York": {"latitude": 40.7128, "longitude": -74.0060},
    "Philadelphia": {"latitude": 39.9526, "longitude": -75.1652},
    "Pittsburgh": {"latitude": 40.4406, "longitude": -79.9959},
    "Scottsdale": {"latitude": 33.4942, "longitude": -111.9261},
    "Seattle": {"latitude": 47.6062, "longitude": -122.3321},
    "St Louis": {"latitude": 38.6270, "longitude": -90.1994},
    "Manchester": {"latitude": 53.4808, "longitude": -2.2426},
    "Boston": {"latitude": 42.3601, "longitude": -71.0589},
    "San Diego": {"latitude": 32.7157, "longitude": -117.1611},
    "Salt Lake City": {"latitude": 40.7608, "longitude": -111.8910},
    "New Orleans": {"latitude": 29.9511, "longitude": -90.0715},
    "Santa Monica": {"latitude": 34.0195, "longitude": -118.4912},
    "Detroit": {"latitude": 42.3314, "longitude": -83.0458},
    "Sydney": {"latitude": -33.8688, "longitude": 151.2093},
    "London": {"latitude": 51.5074, "longitude": -0.1278}
}

cities_weather={}

In [2]:
#looping through cities and for each of them extracting relevant weather data into a dictionary

for city, coordinates in cities_coordinates.items():
    lat = coordinates["latitude"]
    lon = coordinates["longitude"]

    #hourly temp and auto timezone, the goal later will be to filter out the data for 12:00 PM local timezone for each city
    #daily url - url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&timezone=auto&hourly=temperature_2m,precipitation,relative_humidity_2m,precipitation_probability&forecast_days=1"

    #initial url for celsius and inch
    #url=f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&timezone=auto&hourly=temperature_2m,precipitation,relative_humidity_2m,precipitation_probability&precipitation_unit=inch&start_date=2026-05-01&end_date=2026-05-11"

    #initial url for fahrenheit and inch
    #url=f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&timezone=auto&hourly=temperature_2m,precipitation,relative_humidity_2m,precipitation_probability&temperature_unit=fahrenheit&precipitation_unit=inch&start_date=2026-05-12&end_date=2026-06-01"

    #initial url for fahrenheit and cm
    url=f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&timezone=auto&hourly=temperature_2m,precipitation,relative_humidity_2m,precipitation_probability&temperature_unit=fahrenheit&start_date=2026-06-02&end_date=2026-07-07"
    
    try:
        response = requests.get(url)
        response.raise_for_status()  #checking for http errors
        
        cities_weather[city] = response.json()
        print(f"data fetched for {city}")
        
    except Exception as e:
        print(f"fetching failed for {city}: {e}")

print("extraction complete")

data fetched for Atlanta
data fetched for Charlotte
data fetched for Chicago
data fetched for Cleveland
data fetched for Copenhagen
data fetched for Denver
data fetched for Las Vegas
data fetched for Minneapolis
data fetched for New York
data fetched for Philadelphia
data fetched for Pittsburgh
data fetched for Scottsdale
data fetched for Seattle
data fetched for St Louis
data fetched for Manchester
data fetched for Boston
data fetched for San Diego
data fetched for Salt Lake City
data fetched for New Orleans
data fetched for Santa Monica
data fetched for Detroit
data fetched for Sydney
data fetched for London
extraction complete


In [3]:
#in this part we are filtering only the weather for 12PM for each city for all days
#logic is the same as in daily ETL we are just extracting multiple days at the time

cities_weather_filtered = cp.deepcopy(cities_weather)

for city, city_data in cities_weather_filtered.items():
    noon_i = []
    i = 0
    for string in city_data["hourly"]["time"]:
        if string.endswith("T12:00") == True:
            noon_i.append(i) 
        i = i + 1

    helper_hourly = {}
    for key, value in city_data["hourly"].items():
        noon_values_for_key = [] 
        
        for idx in noon_i:
            noon_values_for_key.append(value[idx])

        helper_hourly[key] = noon_values_for_key

    city_data["weather12PM"] = helper_hourly
    del city_data["hourly"]

    

In [4]:
#in this part we are loading data into a dataframe and doing needed changes so that we are left with columns which can be loaded into a csv file

df_start=pd.DataFrame(cities_weather_filtered)
df_T=df_start.T #cities are columns so we need to transpose
df=df_T.reset_index() #reset index so that cities are a column and not an index
df=df.rename(columns={"index": "city", "generationtime_ms": "generation_time_ms","hourly_units":"units","weather12PM":"weather"})
df_units = pd.json_normalize(df['units']).add_suffix('_unit')
df_j1 = df.join(df_units)
df_weather = pd.json_normalize(df['weather'])
df_flat = df_j1.join(df_weather)

weather_cols = list(df_weather.columns)
df_final = df_flat.explode(weather_cols) #explode because in weather column there are lists

df_final = df_final.drop(columns=['units', 'weather'])

df_final = df_final.reset_index(drop=True)


In [5]:
#in this part we are extracting data into a csv file which we will later use as a source for our staging table in the database

date_str=datetime.now().strftime('%Y%m%d')
filename=f"export_{date_str}.csv"

df_csv = df_final.replace({';':''}, regex=True) #just in case because we will use ; as a separator
#Warning could be solved using pd.set_option('future.no_silent_downcasting', True), but it's not used here

df_csv.to_csv(filename, header = True, index = False, sep=';') #if a file already exists data will be overwritten

C:\Users\Matej\AppData\Local\Temp\ipykernel_9476\1019733781.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_csv = df_final.replace({';':''}, regex=True) #just in case because we will use ; as a separator
